# GLNBench quickstart

This notebook runs one small CPU-friendly Cora experiment. It clones the public repository, installs a runtime-compatible quickstart environment, writes a native GLNBench YAML configuration, and starts the benchmark. Google Colab's Python 3.12 runtime uses its existing PyTorch build; the fully pinned Python 3.10/3.11 environment remains available for reproducible full runs.

In [ ]:
import sys

QUICKSTART_PYTHONS = {(3, 10), (3, 11), (3, 12)}
if sys.version_info[:2] not in QUICKSTART_PYTHONS:
    raise RuntimeError(
        f"The GLNBench quickstart supports Python 3.10-3.12; this runtime is {sys.version.split()[0]}. "
        "Use a supported Colab runtime or follow the local setup instructions in the README."
    )
environment = "pinned release" if sys.version_info[:2] in {(3, 10), (3, 11)} else "Colab quickstart"
print(f"Python runtime is compatible: {sys.version.split()[0]} ({environment} environment)")

In [ ]:
import subprocess
import sys

!test -d GLNBench || git clone --depth 1 https://github.com/GLNBench/GLNBench.git
%cd GLNBench

if sys.version_info[:2] == (3, 12):
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError("The Python 3.12 quickstart expects Colab's preinstalled PyTorch.") from exc
    requirements_file = "requirements-colab.txt"
else:
    requirements_file = "requirements.txt"

print(f"Installing {requirements_file} with {sys.executable}")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", requirements_file])

In [ ]:
from pathlib import Path

config = """seed: 42
device: cpu
num_runs: 1

dataset:
  name: cora
  root: data
  normalize: true

noise:
  type: uniform
  rate: 0.2
  seed: 42

model:
  name: gcn
  hidden_channels: 16
  n_layers: 2
  dropout: 0.2
  self_loop: true
  use_residual: false
  jk: none
  normalization: layer

training:
  method: standard
  lr: 0.01
  weight_decay: 0.0005
  epochs: 200
  patience: 20
  early_stopping_metric: val_acc
  oversmoothing_every: 20
  mode: transductive
"""
Path("quickstart.yaml").write_text(config, encoding="utf-8")
print(config)

In [ ]:
!python3 main.py -c quickstart.yaml --results results/quickstart
print("Finished. Inspect results/quickstart for experiment.json and per-run logs.")